# Customer segmentation with K-means

**Goal:** find groups of customers that a marketing team can describe in one sentence and act on.

There is no label to predict, so this is unsupervised learning. The judgment calls are which features to use, how many groups are real, and whether the groups mean anything to a marketer. The reusable code lives in `src/segmentation/`; this notebook walks through the analysis and the reasoning.

**Data:** the public *Customer Personality Analysis* file (2,240 customers, 29 columns). See `data/README.md` for where to get it. It is not committed to this repo.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from segmentation import config
from segmentation.cleaning import clean
from segmentation.data import load_customers
from segmentation.features import cluster_matrix, scale
from segmentation.modeling import (
    elbow_inertia,
    fit_kmeans,
    name_segments,
    pca_cumulative_variance,
    profile,
    silhouette_by_k,
)
from segmentation.synthetic import make_customers

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Seed:", config.SEED, "| Reference year for ages:", config.REFERENCE_YEAR)

## Load the data

Set `SEGMENTATION_SYNTHETIC=1` to run the notebook on generated data (CI does this to prove the notebook runs). Results only mean something on the real file.

In [ ]:
DATA_PATH = Path(os.environ.get("SEGMENTATION_DATA", "../data/raw/marketing_campaign.csv"))
SYNTHETIC = os.environ.get("SEGMENTATION_SYNTHETIC") == "1"

raw = make_customers() if SYNTHETIC else load_customers(DATA_PATH)
print(raw.shape)
raw.info()

In [ ]:
raw.describe().T

What stands out before any cleaning (original run):

- **Income** has 24 missing values and an extreme maximum (666,666), so a few records pull the mean up.
- **Year of birth** goes back to 1893, which is not plausible for a living customer.
- **Z_CostContact** and **Z_Revenue** are constant, so they carry no information.
- Spending is heavily right-skewed; wine and meat dominate.

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
raw["Marital_Status"].value_counts()

## Clean the data

Each rule is a small function in `segmentation/cleaning.py` with a unit test, and the cleaning step returns a report so every fix is counted.

| Problem | Fix | Why |
|---|---|---|
| Missing income | Fill with the median | Outliers pull the mean; the median is robust |
| Birth year | Convert to age at a fixed reference year | Age is easier to read; a fixed year keeps results reproducible |
| Ages over 100 | Replace with the rounded mean of valid ages | Keeps those customers' spending data, which matters more for clustering |
| "Alone" | Map to Single | Same meaning |
| "Absurd", "YOLO" | Map to the most common status | Not real categories; only 4 records |

In [ ]:
df, report = clean(raw)
report

## Explore

Distributions of the clustering features after cleaning, then correlations.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(14, 16))
for ax, col in zip(axes.flat, config.CLUSTER_FEATURES, strict=True):
    sns.histplot(df[col], ax=ax, kde=True, bins=20)
    ax.axvline(df[col].mean(), color="green", linestyle="--", label="mean")
    ax.axvline(df[col].median(), color="black", label="median")
    ax.set_title(col)
axes.flat[0].legend()
plt.tight_layout()
plt.show()

Income and age look roughly normal once the bad ages are fixed. Spending and catalog purchases are right-skewed: most customers spend little and a minority spend a lot.

In [ ]:
corr_cols = config.CLUSTER_FEATURES + ["Recency", "Days_Customer"]
plt.figure(figsize=(15, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="RdBu_r")
plt.title("Correlation of numeric features")
plt.show()

What I took from the correlations:

- Spending rises with income and with catalog and store purchases.
- Customers with young children at home visit the website more and spend less.
- **Recency** and **days as a customer** have almost no correlation with anything else, so I left them out of the clustering features. That leaves 15 features.

## Scale

K-means uses distances, so a feature measured in dollars would dominate one measured in counts. Z-scoring puts every feature on the same footing.

In [ ]:
features = cluster_matrix(df)
scaled, scaler = scale(features)
scaled.describe().T[["mean", "std"]].round(3)

## Does PCA help?

PCA could compress the 15 features before clustering. It only helps if a few components keep most of the variance.

In [ ]:
cumvar = pca_cumulative_variance(scaled)
plt.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
plt.axhline(0.9, color="grey", linestyle="--")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.show()
print(f"11 components keep {cumvar[10]:.1%} of the variance")

In the original run, keeping 92.4% of the variance needed 11 of 15 components. That is very little compression, and clusters built on components are hard to explain to a marketer. **Decision: cluster on the scaled features, not on PCA components.**

## Choose the number of clusters

Two methods: the elbow plot (where adding clusters stops paying off) and the silhouette score (how well separated the clusters are).

In [ ]:
inertia = elbow_inertia(scaled)
silhouette = silhouette_by_k(scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(inertia), list(inertia.values()), "o-")
ax1.set(title="Elbow plot", xlabel="k", ylabel="Within-cluster sum of squares")
ax2.plot(list(silhouette), list(silhouette.values()), "o-")
ax2.set(title="Silhouette score", xlabel="k", ylabel="Mean silhouette")
plt.tight_layout()
plt.show()
pd.Series(silhouette, name="silhouette").round(4)

The elbow plot leaves k = 2 or 3 open. The silhouette score settles it: **k = 2** has the best separation. Two segments also means two campaigns a team can actually run.

## Fit the final model

In [ ]:
model = fit_kmeans(scaled, k=2)
profiles = profile(scaled, model.labels_)
names = name_segments(profiles)
segments = pd.Series(model.labels_).map(names)
segments.value_counts()

## Profile the segments

Mean scaled values: 0 is the average customer; positive is above average.

In [ ]:
profiles.rename(index=names).T.round(2)

In [ ]:
ax = profiles.rename(index=names).T.plot(kind="barh", figsize=(10, 9))
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Mean scaled feature value by segment", xlabel="Mean scaled value")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = scaled.assign(segment=segments.values)
fig, axes = plt.subplots(5, 3, figsize=(14, 18))
for ax, col in zip(axes.flat, config.CLUSTER_FEATURES, strict=True):
    sns.boxplot(data=plot_df, x="segment", y=col, ax=ax, showfliers=False)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## What the segments mean

**Segment A: established, high-spending households** (922 customers in the original run). Well above average income; above-average spending in every category, furthest on wine and meat; the most catalog and store purchases; fewer web visits and deal purchases; fewer young children; slightly older.

**Segment B: younger families** (1,318 customers). Below-average income and spending; more web visits; slightly more deal purchases; more young children at home.

## What I'd tell marketing

- **Segment A** buys across every channel and rarely waits for a deal. Don't train them to expect discounts. Invest in retention, premium ranges (wine and meat), and catalog and in-store experiences.
- **Segment B** visits the website often but spends less and responds more to deals. Target them online with family-oriented offers and web-only deals, and measure whether the visits turn into larger baskets.

## What I'd do next

- Test k = 3 and Gaussian mixture models to see whether Segment B splits into groups worth separate campaigns.
- Validate the segments against past campaign responses, which I deliberately left out of the clustering, before spending money on them.
- Compute tenure from a fixed date as well, so every derived feature is reproducible.